# Noise and Error Mitigation

Model NISQ noise yourself in NumPy with density matrices, then meet zero-noise extrapolation.

**Objectives:**
- Lift a pure state from `circuit.state_vector()` into a density matrix `rho = |psi><psi|`
- Apply the depolarizing and amplitude-damping channels as Kraus sums on `rho`
- Quantify decay with the fidelity `F = <psi|rho|psi>` and watch a clean distribution rot toward noise
- Build the conceptual core of zero-noise extrapolation (ZNE) and see why `DM1` exists

**Reference:** See [`../GUIDE.md`](../GUIDE.md) for concept explanations and context.

<!-- browser-runnable -->

In [ ]:
# Setup: run this cell first.
# qcsim is a pure-state simulator with NO built-in noise and NO density-matrix device.
# So we model noise EXACTLY, ourselves, in NumPy -- exactly what the managed DM1
# simulator does at scale. Everything here runs locally and spends nothing.

from braket.circuits import Circuit
from braket.devices import LocalSimulator
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
np.set_printoptions(precision=4, suppress=True)

device = LocalSimulator()
print("Ready. We will build density matrices in NumPy from circuit.state_vector().")

## 1. From a pure state to a density matrix

The state vector `|psi>` you have used all along is a *pure* state. A perfect, noiseless
machine stays pure. Real hardware does not: it produces **mixed** states -- statistical blends
of pure states -- and a bare vector cannot represent those. The density matrix can.

For a pure state the density matrix is just the outer product

$$\rho = |\psi\rangle\langle\psi| = \text{outer}(\text{sv},\ \overline{\text{sv}}).$$

A valid density matrix always has $\operatorname{Tr}(\rho) = 1$ (probabilities sum to one) and is
Hermitian. We build it straight from `circuit.state_vector()` -- qcsim is big-endian (qubit 0 is
the leftmost/most-significant bit), and the vector spans all qubits the circuit touches.

In [ ]:
# The Bell pair from foundations: H on qubit 0, then CNOT(0 -> 1).
# Big-endian, so the ideal state is (|00> + |11>)/sqrt(2).
bell = Circuit().h(0).cnot(0, 1)
sv_ideal = bell.state_vector()           # full complex numpy vector, length 2**2 = 4
rho_ideal = np.outer(sv_ideal, sv_ideal.conj())

print("ideal state vector (big-endian |00>,|01>,|10>,|11>):")
print(sv_ideal)
print("\nideal density matrix rho = |psi><psi|:")
print(rho_ideal)
print("\nTr(rho) =", np.round(np.trace(rho_ideal).real, 12))

assert abs(np.trace(rho_ideal).real - 1.0) < 1e-12, "density matrix must have unit trace"
assert np.allclose(rho_ideal, rho_ideal.conj().T), "density matrix must be Hermitian"

## 2. Embedding single-qubit operators on n qubits

A noise channel acts on one physical qubit, but `rho` lives on the whole register. To apply a
single-qubit operator $O$ to qubit $q$ of an $n$-qubit state we embed it with Kronecker products:
identity on every qubit except $q$. Because qcsim is **big-endian**, qubit 0 is the *leftmost*
factor in the `kron`, so we build left-to-right from qubit 0 to qubit $n-1$.

We will need the Pauli matrices $X, Y, Z$ for depolarizing noise, so we define them once.

In [ ]:
I2 = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)


def embed(op, q, n):
    """Embed single-qubit operator `op` on qubit q of an n-qubit register (big-endian)."""
    m = np.array([[1.0]], dtype=complex)
    for i in range(n):
        m = np.kron(m, op if i == q else I2)
    return m


# Sanity: Z on qubit 0 of 2 qubits = Z (x) I = diag(1, 1, -1, -1) big-endian.
Z0 = embed(Z, 0, 2)
print("Z on qubit 0 of 2 (diagonal):", np.round(np.diag(Z0).real, 0))
assert np.allclose(np.diag(Z0), np.array([1, 1, -1, -1], dtype=complex))
assert np.allclose(embed(I2, 0, 1), I2)  # trivial embedding

## 3. The depolarizing channel

Depolarizing noise nudges a qubit toward the maximally mixed state -- a coin that forgets which
way it was leaning. With probability $1-p$ nothing happens; with probability $p$ the qubit is hit
by a uniformly random Pauli error. On qubit $q$:

$$\rho \;\longrightarrow\; (1-p)\,\rho \;+\; \frac{p}{3}\big(X_q\,\rho\,X_q + Y_q\,\rho\,Y_q + Z_q\,\rho\,Z_q\big),$$

where $X_q, Y_q, Z_q$ are the embedded Paulis from the previous section. This is a Kraus sum;
because the Paulis are unitary and the weights $\{1-p, p/3, p/3, p/3\}$ sum to 1, it preserves the
trace -- noise reshuffles probability, it never destroys it.

In [ ]:
def depolarizing(rho, q, n, p):
    """Single-qubit depolarizing channel on qubit q of an n-qubit density matrix."""
    Xq, Yq, Zq = embed(X, q, n), embed(Y, q, n), embed(Z, q, n)
    return (1 - p) * rho + (p / 3.0) * (Xq @ rho @ Xq + Yq @ rho @ Yq + Zq @ rho @ Zq)


# p = 0 must be a no-op: the channel leaves rho exactly unchanged.
assert np.allclose(depolarizing(rho_ideal, 0, 2, 0.0), rho_ideal)

# Trace is preserved for any p.
for p in [0.1, 0.4, 0.75]:
    r = depolarizing(rho_ideal, 0, 2, p)
    assert abs(np.trace(r).real - 1.0) < 1e-12
print("depolarizing channel: p=0 is a no-op and Tr(rho)=1 for all p. OK")

## 4. Full depolarization reaches the maximally mixed state

How strong is "fully depolarized"? Take a single qubit in $|+\rangle$ and push $p$ up. A common
intuition says $p=1$ gives the maximally mixed state $I/2$ -- but watch the algebra of *this exact*
channel. With $X|+\rangle = |+\rangle$, $Z|+\rangle = |-\rangle$, $Y|+\rangle = i|-\rangle$, the
$p=1$ output is $\tfrac{1}{3}(|+\rangle\langle+| + 2|-\rangle\langle-|) = (2I-\rho)/3$, which still
carries off-diagonal structure -- it is *not* $I/2$.

The maximally mixed point for the $p/3$ convention is actually $p = 3/4$, where the four operators
$\{\rho, X\rho X, Y\rho Y, Z\rho Z\}$ enter with equal weight $1/4$ and average to $I/2$ by the
Pauli completeness identity. There the state has forgotten everything: fidelity to *any* pure
state is exactly $1/2$ for one qubit. This is a real, load-bearing subtlety of the channel's
convention, not a rounding artifact.

In [ ]:
plus = Circuit().h(0).state_vector()       # |+> on a single qubit
rho_plus = np.outer(plus, plus.conj())

rho_max = depolarizing(rho_plus, 0, 1, 0.75)
print("depolarizing |+> at p = 3/4:")
print(rho_max)

# At p = 3/4 the exact (p/3)-convention channel gives the maximally mixed state I/2.
assert np.allclose(rho_max, I2 / 2), "p=3/4 must give the maximally mixed state I/2"

# Fidelity to ANY pure state is then exactly 1/2. Check |+> and |0>.
zero = np.array([1, 0], dtype=complex)
F_plus = (plus.conj() @ rho_max @ plus).real
F_zero = (zero.conj() @ rho_max @ zero).real
print("\nfidelity of I/2 to |+>:", round(float(F_plus), 6))
print("fidelity of I/2 to |0>:", round(float(F_zero), 6))
assert abs(F_plus - 0.5) < 1e-12 and abs(F_zero - 0.5) < 1e-12
print("\nFully depolarized -> fidelity 0.5 to any pure state. OK")

## 5. Amplitude damping -- energy loss

Depolarizing is symmetric, but real qubits decay in a *direction*: an excited $|1\rangle$ relaxes
toward the ground state $|0\rangle$ as it leaks energy to the environment ($T_1$ decay). The
amplitude-damping channel captures this with two Kraus operators parameterized by the damping
probability $\gamma$:

$$K_0 = \begin{pmatrix}1 & 0\\ 0 & \sqrt{1-\gamma}\end{pmatrix}, \qquad K_1 = \begin{pmatrix}0 & \sqrt{\gamma}\\ 0 & 0\end{pmatrix}, \qquad \rho \longrightarrow K_0\,\rho\,K_0^\dagger + K_1\,\rho\,K_1^\dagger.$$

$K_1$ is the actual decay event ($|1\rangle \to |0\rangle$); $K_0$ is the "no decay" branch, which
still shrinks the $|1\rangle$ amplitude. Together $K_0^\dagger K_0 + K_1^\dagger K_1 = I$, so the
trace is preserved. At $\gamma = 1$ the excited state has fully relaxed: $|1\rangle \to |0\rangle$.

In [ ]:
def amplitude_damping(rho, q, n, g):
    """Amplitude-damping channel with damping probability g on qubit q."""
    K0 = np.array([[1, 0], [0, np.sqrt(1 - g)]], dtype=complex)
    K1 = np.array([[0, np.sqrt(g)], [0, 0]], dtype=complex)
    K0q, K1q = embed(K0, q, n), embed(K1, q, n)
    return K0q @ rho @ K0q.conj().T + K1q @ rho @ K1q.conj().T


# Prepare |1> and fully damp it: it must land exactly on |0><0|.
one = Circuit().x(0).state_vector()
rho_one = np.outer(one, one.conj())
rho_damped = amplitude_damping(rho_one, 0, 1, 1.0)

print("amplitude damping |1> at g = 1:")
print(rho_damped)
rho_zero = np.outer(zero, zero.conj())
assert np.allclose(rho_damped, rho_zero), "g=1 must drive |1> to |0>"

# g = 0 is a no-op; trace preserved for all g.
assert np.allclose(amplitude_damping(rho_one, 0, 1, 0.0), rho_one)
for g in [0.2, 0.5, 0.9]:
    assert abs(np.trace(amplitude_damping(rho_one, 0, 1, g)).real - 1.0) < 1e-12
print("\ng=1 sends |1> -> |0>; g=0 is a no-op; Tr(rho)=1 throughout. OK")

## 6. Fidelity: how much of the ideal survives

**Fidelity** measures how close the noisy state stays to the ideal pure state $|\psi\rangle$. For a
pure reference it is simply the expectation of $\rho$ in that state:

$$F = \langle\psi|\,\rho\,|\psi\rangle = \operatorname{Re}\big(\overline{\text{sv}}\cdot \rho\cdot \text{sv}\big).$$

It runs from 1 (the noisy state is still exactly $|\psi\rangle$) down toward 0 (all signal lost).
This single number is how we will watch noise eat a circuit.

In [ ]:
def fidelity(sv, rho):
    """Fidelity of density matrix rho to the ideal pure state sv."""
    return float((sv.conj() @ rho @ sv).real)


# With no noise, fidelity to its own ideal state is exactly 1.
assert abs(fidelity(sv_ideal, rho_ideal) - 1.0) < 1e-12

# A little depolarizing on both Bell qubits should drop fidelity below 1.
rho_noisy = rho_ideal.copy()
for q in range(2):
    rho_noisy = depolarizing(rho_noisy, q, 2, 0.1)
print("ideal fidelity:", fidelity(sv_ideal, rho_ideal))
print("fidelity after p=0.1 depolarizing on both qubits:", round(fidelity(sv_ideal, rho_noisy), 4))
assert fidelity(sv_ideal, rho_noisy) < 1.0

## 7. A clean distribution rotting toward noise

Now make it visible. The ideal Bell pair has two crisp peaks at $|00\rangle$ and $|11\rangle$ and
zero everywhere else. As depolarizing noise rises, the diagonal of $\rho$ -- the measurement
probabilities -- smears toward flat. We compare the ideal sampled counts (qcsim, real shots) with
the noisy probabilities we read off the diagonal of $\rho$.

In [ ]:
labels = ["00", "01", "10", "11"]

# Ideal distribution, sampled on qcsim (big-endian, all qubits measured at once).
counts = device.run(bell, shots=2000).result().measurement_counts
ideal_probs = np.array([counts.get(b, 0) for b in labels]) / 2000.0


def noisy_probs(p):
    """Measurement probabilities = diagonal of rho after depolarizing both qubits at rate p."""
    r = rho_ideal.copy()
    for q in range(2):
        r = depolarizing(r, q, 2, p)
    return np.diag(r).real, r


for p in [0.0, 0.2, 0.5, 0.75]:
    pr, r = noisy_probs(p)
    print(f"p={p:.2f}  probs={np.round(pr, 3)}  fidelity={round(fidelity(sv_ideal, r), 4)}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
x = np.arange(4)

ax[0].bar(x, ideal_probs, color="#2563eb")
ax[0].set_title("Ideal Bell pair (sampled, p = 0)")
ax[0].set_xticks(x); ax[0].set_xticklabels(labels)
ax[0].set_ylabel("probability"); ax[0].set_ylim(0, 0.6)

width = 0.27
for i, p in enumerate([0.2, 0.5, 0.75]):
    pr, _ = noisy_probs(p)
    ax[1].bar(x + (i - 1) * width, pr, width, label=f"p={p}")
ax[1].axhline(0.25, ls="--", color="gray", lw=1, label="flat noise (0.25)")
ax[1].set_title("Depolarized: peaks rot toward flat")
ax[1].set_xticks(x); ax[1].set_xticklabels(labels)
ax[1].set_ylim(0, 0.6); ax[1].legend(fontsize=8)

plt.tight_layout(); plt.show()
print("At p=3/4 the two-qubit state is maximally mixed: every outcome -> 0.25.")

## 8. Fidelity vs error rate

Plot the single decay curve that summarizes it all. We sweep the depolarizing rate on the Bell
pair over the **physical noise regime** $p \in [0, 3/4]$ -- from clean to maximally mixed -- and
the amplitude-damping rate on a single $|1\rangle$ over $\gamma \in [0, 1]$.

A note on the depolarizing range: past $p = 3/4$ this $p/3$-convention channel *over-rotates* and
fidelity climbs again (the $(2I-\rho)/3$ behavior we saw in section 4). The genuinely physical,
monotone decay lives on $[0, 3/4]$, so that is where we assert monotonicity.

In [ ]:
# Depolarizing sweep on the Bell pair (both qubits), p in [0, 0.75].
ps = np.linspace(0.0, 0.75, 16)
depol_fids = []
for p in ps:
    r = rho_ideal.copy()
    for q in range(2):
        r = depolarizing(r, q, 2, p)
    assert abs(np.trace(r).real - 1.0) < 1e-12          # trace preserved everywhere
    depol_fids.append(fidelity(sv_ideal, r))
depol_fids = np.array(depol_fids)

# Amplitude-damping sweep on |1>, gamma in [0, 1], fidelity measured to the ideal |1>.
gs = np.linspace(0.0, 1.0, 16)
amp_fids = []
for g in gs:
    r = amplitude_damping(rho_one, 0, 1, g)
    assert abs(np.trace(r).real - 1.0) < 1e-12
    amp_fids.append(fidelity(one, r))
amp_fids = np.array(amp_fids)

# Asserts: clean start, and monotonically NON-increasing decay.
assert abs(depol_fids[0] - 1.0) < 1e-12 and abs(amp_fids[0] - 1.0) < 1e-12
assert np.all(np.diff(depol_fids) <= 1e-12), "depolarizing fidelity must not increase on [0, 3/4]"
assert np.all(np.diff(amp_fids) <= 1e-12), "amplitude-damping fidelity must not increase"
print("depol fidelity 1.00 ->", round(depol_fids[-1], 3), " (maximally mixed: 1/4 for 2 qubits)")
print("amp-damp fidelity 1.00 ->", round(amp_fids[-1], 3), " (|1> fully relaxed to |0>)")

In [ ]:
plt.figure(figsize=(7, 4.2))
plt.plot(ps, depol_fids, "o-", color="#dc2626", label="depolarizing (Bell, both qubits)")
plt.plot(gs, amp_fids, "s-", color="#2563eb", label="amplitude damping (|1>)")
plt.axhline(0.25, ls="--", color="gray", lw=1, label="2-qubit maximally mixed (0.25)")
plt.xlabel("error rate  (p  or  gamma)")
plt.ylabel("fidelity to ideal state")
plt.title("Fidelity decays as the error rate rises")
plt.ylim(0, 1.02); plt.legend(fontsize=8); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 9. Zero-noise extrapolation (ZNE), conceptually

You cannot turn a real machine's noise *off*. But you can usually turn it *up* -- by stretching
gates or folding the circuit -- and run the same computation at several known noise levels. If you
then plot your result against the noise scale and extrapolate the trend back to **zero noise**, you
estimate the noise-free answer without ever achieving noise-free hardware. That is **zero-noise
extrapolation**, one of the cheapest and most-used error-mitigation tricks of the NISQ era.

Here we emulate it: take a base depolarizing rate, run at scale factors $1\times, 2\times, 3\times$,
fit a line through the noisy fidelities, and read off its intercept at scale $0$. The extrapolated
value lands above the raw $1\times$ measurement and close to the true fidelity of 1 -- mitigation
recovered signal we could not measure directly.

In [ ]:
def bell_fidelity_at(p):
    r = rho_ideal.copy()
    for q in range(2):
        r = depolarizing(r, q, 2, p)
    return fidelity(sv_ideal, r)


base_p = 0.03
scales = np.array([1.0, 2.0, 3.0])
noisy = np.array([bell_fidelity_at(base_p * s) for s in scales])

# Linear fit fidelity ~ a*scale + b; the intercept b is the scale->0 estimate.
A = np.vstack([scales, np.ones_like(scales)]).T
slope, intercept = np.linalg.lstsq(A, noisy, rcond=None)[0]

print("scale factors      :", scales.tolist())
print("noisy fidelities   :", np.round(noisy, 4).tolist())
print("raw 1x measurement :", round(float(noisy[0]), 4))
print("ZNE estimate (s->0):", round(float(intercept), 4), "  (true noiseless = 1.0)")

# ZNE must improve on the raw measurement and approach the ideal.
assert intercept > noisy[0], "extrapolation should beat the raw noisy result"
assert abs(intercept - 1.0) < 0.05, "extrapolation should land near the ideal fidelity"

In [ ]:
plt.figure(figsize=(6.5, 4))
plt.plot(scales, noisy, "o", color="#dc2626", ms=8, label="measured (noisy)")
xs = np.linspace(0, 3, 50)
plt.plot(xs, slope * xs + intercept, "-", color="#2563eb", label="linear fit")
plt.plot(0, intercept, "*", color="#16a34a", ms=16, label="ZNE estimate (scale=0)")
plt.axhline(1.0, ls="--", color="gray", lw=1, label="true noiseless = 1.0")
plt.xlabel("noise scale factor"); plt.ylabel("fidelity")
plt.title("Zero-noise extrapolation")
plt.legend(fontsize=8); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 10. Why DM1 exists

Everything above ran in NumPy on tiny states. The trade-off the GUIDE names is exactly this: a
density matrix for $n$ qubits is $2^n \times 2^n$ -- it squares the memory a state vector needs, so
doing this by hand stops being feasible quickly. **DM1** is Amazon Braket's managed
density-matrix simulator: it applies these same noise channels (depolarizing, amplitude damping,
and more) on up to 17 qubits, charging per minute. It is the rung of the simulator ladder where
you *study noise* -- after developing on the local simulator and validating scale on SV1, and
**before** paying a real QPU to show you the same decay. `SV1` cannot do this: it is an exact,
noiseless state-vector simulator with no way to represent a mixed state.

The discipline holds: develop on Local, validate on SV1, study noise on DM1, run on a QPU only
when the algorithm is proven.

## Exercises

Try these by extending the channels and helpers above. All run locally in NumPy.

1. **Phase damping.** Implement the dephasing channel with Kraus operators
   `K0 = diag(1, sqrt(1-g))`, `K1 = diag(0, sqrt(g))`. Confirm it kills off-diagonal coherences of
   `|+>` while leaving the populations on the diagonal untouched (unlike amplitude damping).
   ```python
   # def phase_damping(rho, q, n, g):
   #     K0 = np.array([[1, 0], [0, np.sqrt(1 - g)]], dtype=complex)
   #     K1 = np.array([[0, 0], [0, np.sqrt(g)]], dtype=complex)
   #     ...
   # rho_plus_dephased = phase_damping(np.outer(plus, plus.conj()), 0, 1, 1.0)
   # assert np.allclose(np.diag(rho_plus_dephased).real, [0.5, 0.5])
   ```

2. **Per-gate noise.** Apply a small depolarizing kick after *every* gate of a 1-qubit circuit
   `Circuit().h(0).h(0)` (which should return |0>) and plot fidelity vs the number of gates.
   Confirm deeper circuits lose more fidelity -- the NISQ depth penalty.
   ```python
   # for depth in range(1, 11):
   #     rho = ...   # alternate H and a p-depolarizing step, `depth` times
   #     fids.append(fidelity(target_sv, rho))
   ```

3. **Non-linear ZNE.** Re-run section 9 with a larger `base_p` (say 0.15) where the fidelity-vs-
   scale trend curves. Compare a linear fit against a quadratic `np.polyfit(scales, noisy, 2)`
   extrapolation to scale 0. Which gets closer to 1.0, and why?
   ```python
   # coeffs = np.polyfit(scales, noisy, 2)
   # est = np.polyval(coeffs, 0.0)
   ```

## Summary

- A **density matrix** `rho = outer(sv, conj(sv))` represents mixed (noisy) states that a bare
  state vector cannot; valid ones satisfy `Tr(rho) = 1` and are Hermitian.
- The **depolarizing** channel `(1-p)rho + (p/3)(XrhoX + YrhoY + ZrhoZ)` pushes a qubit toward the
  maximally mixed state; for this `p/3` convention the fully-mixed point `I/2` is reached at
  `p = 3/4` (fidelity 0.5 to any pure state), not at `p = 1`.
- **Amplitude damping** `K0 rho K0^dag + K1 rho K1^dag` models energy loss; at `g = 1` the excited
  `|1>` relaxes exactly to `|0>`.
- **Fidelity** `F = <psi|rho|psi>` is 1 with no noise and decays monotonically across the physical
  noise regime -- the clean Bell peaks rot toward flat 0.25.
- **Zero-noise extrapolation** runs the computation at several amplified noise levels and
  extrapolates back to zero noise, recovering signal you cannot measure directly.
- **DM1** is the managed density-matrix simulator that does all of this at scale; `SV1` (noiseless
  state vector) cannot represent mixed states at all.

**Next:** You have finished `02-hardware`. Continue to [`../../03-algorithms/GUIDE.md`](../../03-algorithms/GUIDE.md) to put these noiseless-then-validated circuits to work -- Deutsch-Jozsa, Grover, QFT, and QAOA.